<a href="https://colab.research.google.com/github/tony0990/Data_mining/blob/main/Dashboard_data_mining.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install plotly -q

In [3]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
import networkx as nx
import json

df = pd.read_csv("repository_link_analysis_results.csv")

# ---------- Prepare Data ----------
metrics = ["degree_centrality", "pagerank_score", "hub_score", "authority_score"]

top_pagerank = df.sort_values("pagerank_score", ascending=False).head(15)
top_degree = df.sort_values("degree_centrality", ascending=False).head(15)
top_authority = df.sort_values("authority_score", ascending=False).head(15)
top_hub = df.sort_values("hub_score", ascending=False).head(15)

top1_pagerank = df.sort_values("pagerank_score", ascending=False).iloc[0]
top1_degree = df.sort_values("degree_centrality", ascending=False).iloc[0]
top1_authority = df.sort_values("authority_score", ascending=False).iloc[0]
top1_hub = df.sort_values("hub_score", ascending=False).iloc[0]

# ---------- Charts ----------
dark_template = "plotly_dark"

fig_pagerank = px.bar(
    top_pagerank,
    x="pagerank_score",
    y="repository",
    orientation="h",
    color="pagerank_score",
    color_continuous_scale="Viridis",
    title="Top 15 Repositories by PageRank"
)
fig_pagerank.update_layout(template=dark_template, yaxis=dict(autorange="reversed"), height=430)

fig_degree = px.bar(
    top_degree,
    x="degree_centrality",
    y="repository",
    orientation="h",
    color="degree_centrality",
    color_continuous_scale="Blues",
    title="Top 15 Repositories by Degree Centrality"
)
fig_degree.update_layout(template=dark_template, yaxis=dict(autorange="reversed"), height=430)

fig_scatter = px.scatter(
    df,
    x="degree_centrality",
    y="pagerank_score",
    color="pagerank_score",
    hover_data=["repository"],
    color_continuous_scale="Viridis",
    title="Degree Centrality vs PageRank"
)
fig_scatter.update_layout(template=dark_template, height=430)

fig_hits = px.scatter(
    df,
    x="hub_score",
    y="authority_score",
    color="authority_score",
    hover_data=["repository"],
    color_continuous_scale="Plasma",
    title="HITS Analysis: Hub Score vs Authority Score"
)
fig_hits.update_layout(template=dark_template, height=430)

corr = df[metrics].corr()
fig_corr = px.imshow(
    corr,
    text_auto=True,
    color_continuous_scale="Viridis",
    title="Correlation Matrix Between Graph Metrics"
)
fig_corr.update_layout(template=dark_template, height=430)

# ---------- Network Graph ----------
top_nodes = top_pagerank["repository"].tolist()

G_dash = nx.Graph()
for repo in top_nodes:
    G_dash.add_node(repo)

for i in range(len(top_nodes)):
    for j in range(i + 1, len(top_nodes)):
        r1 = df[df["repository"] == top_nodes[i]].iloc[0]
        r2 = df[df["repository"] == top_nodes[j]].iloc[0]
        diff = abs(r1["pagerank_score"] - r2["pagerank_score"])
        if diff < 0.00005:
            G_dash.add_edge(top_nodes[i], top_nodes[j])

pos = nx.spring_layout(G_dash, seed=42)

edge_x, edge_y = [], []
for edge in G_dash.edges():
    x0, y0 = pos[edge[0]]
    x1, y1 = pos[edge[1]]
    edge_x += [x0, x1, None]
    edge_y += [y0, y1, None]

node_x, node_y, labels, colors = [], [], [], []
for node in G_dash.nodes():
    x, y = pos[node]
    node_x.append(x)
    node_y.append(y)
    labels.append(node)
    colors.append(float(df[df["repository"] == node]["pagerank_score"].iloc[0]))

fig_network = go.Figure()

fig_network.add_trace(go.Scatter(
    x=edge_x,
    y=edge_y,
    mode="lines",
    line=dict(width=1, color="rgba(148, 163, 184, 0.45)"),
    hoverinfo="none"
))

fig_network.add_trace(go.Scatter(
    x=node_x,
    y=node_y,
    mode="markers+text",
    text=labels,
    textposition="top center",
    marker=dict(
        size=22,
        color=colors,
        colorscale="Viridis",
        showscale=True,
        colorbar=dict(title="PageRank")
    ),
    textfont=dict(size=10, color="#e5e7eb"),
    hovertemplate="<b>%{text}</b><br>PageRank=%{marker.color:.6f}<extra></extra>"
))

fig_network.update_layout(
    template=dark_template,
    title="Interactive Repository Network Graph",
    height=520,
    showlegend=False,
    margin=dict(l=10, r=10, t=50, b=10),
    xaxis=dict(showgrid=False, zeroline=False, visible=False),
    yaxis=dict(showgrid=False, zeroline=False, visible=False)
)

# ---------- Table ----------
table_df = top_pagerank[["repository", "degree_centrality", "pagerank_score", "hub_score", "authority_score"]].copy()
table_rows = ""
for _, row in table_df.iterrows():
    table_rows += f"""
    <tr>
        <td>{row['repository']}</td>
        <td>{row['degree_centrality']:.6f}</td>
        <td>{row['pagerank_score']:.6f}</td>
        <td>{row['hub_score']:.6f}</td>
        <td>{row['authority_score']:.6f}</td>
    </tr>
    """

# ---------- HTML ----------
html = f"""
<!DOCTYPE html>
<html>
<head>
<meta charset="UTF-8">
<title>GitHub Repository Mining Dashboard</title>
<script src="https://cdn.plot.ly/plotly-latest.min.js"></script>

<style>
* {{
    box-sizing: border-box;
}}

body {{
    margin: 0;
    font-family: Inter, Arial, sans-serif;
    background: #0f172a;
    color: #e5e7eb;
}}

.sidebar {{
    position: fixed;
    top: 0;
    left: 0;
    width: 260px;
    height: 100vh;
    background: #020617;
    padding: 24px 18px;
    border-right: 1px solid #1e293b;
}}

.logo {{
    font-size: 22px;
    font-weight: 800;
    color: #38bdf8;
    margin-bottom: 8px;
}}

.subtitle {{
    font-size: 13px;
    color: #94a3b8;
    margin-bottom: 30px;
}}

.nav-btn {{
    width: 100%;
    padding: 12px 14px;
    margin-bottom: 10px;
    background: transparent;
    color: #cbd5e1;
    border: 1px solid #1e293b;
    border-radius: 12px;
    cursor: pointer;
    text-align: left;
    font-size: 14px;
}}

.nav-btn:hover, .nav-btn.active {{
    background: linear-gradient(135deg, #0ea5e9, #6366f1);
    color: white;
}}

.main {{
    margin-left: 260px;
    padding: 26px;
}}

.header {{
    display: flex;
    justify-content: space-between;
    align-items: center;
    margin-bottom: 24px;
}}

.header h1 {{
    margin: 0;
    font-size: 30px;
}}

.header p {{
    color: #94a3b8;
    margin-top: 6px;
}}

.badge {{
    background: rgba(14, 165, 233, 0.15);
    color: #38bdf8;
    border: 1px solid rgba(56, 189, 248, 0.4);
    padding: 10px 16px;
    border-radius: 999px;
    font-size: 13px;
}}

.cards {{
    display: grid;
    grid-template-columns: repeat(4, 1fr);
    gap: 16px;
    margin-bottom: 20px;
}}

.card {{
    background: #111827;
    border: 1px solid #1f2937;
    border-radius: 18px;
    padding: 18px;
    box-shadow: 0 10px 30px rgba(0,0,0,0.25);
}}

.card p {{
    margin: 0;
    color: #94a3b8;
    font-size: 13px;
}}

.card h2 {{
    margin: 10px 0 0;
    font-size: 28px;
    color: #38bdf8;
}}

.section {{
    display: none;
}}

.section.active {{
    display: block;
}}

.grid-2 {{
    display: grid;
    grid-template-columns: 1fr 1fr;
    gap: 18px;
}}

.panel {{
    background: #111827;
    border: 1px solid #1f2937;
    border-radius: 18px;
    padding: 18px;
    margin-bottom: 18px;
}}

.panel h2 {{
    margin-top: 0;
    color: #f8fafc;
}}

.insight {{
    background: rgba(56, 189, 248, 0.08);
    border-left: 4px solid #38bdf8;
    padding: 14px;
    border-radius: 12px;
    color: #cbd5e1;
    margin-top: 12px;
    line-height: 1.6;
}}

.top-card {{
    background: linear-gradient(135deg, rgba(14,165,233,0.18), rgba(99,102,241,0.18));
    border: 1px solid rgba(56,189,248,0.35);
    border-radius: 18px;
    padding: 20px;
    margin-bottom: 16px;
}}

.top-card h3 {{
    margin-top: 0;
    color: #38bdf8;
}}

.search-box {{
    width: 100%;
    padding: 13px;
    border-radius: 12px;
    background: #020617;
    color: #e5e7eb;
    border: 1px solid #334155;
    margin-bottom: 14px;
}}

table {{
    width: 100%;
    border-collapse: collapse;
    font-size: 13px;
}}

th {{
    background: #0f172a;
    color: #38bdf8;
    padding: 12px;
    text-align: left;
    border-bottom: 1px solid #334155;
}}

td {{
    padding: 11px;
    border-bottom: 1px solid #1f2937;
    color: #cbd5e1;
}}

tr:hover {{
    background: rgba(56, 189, 248, 0.06);
}}

.footer {{
    text-align: center;
    color: #64748b;
    padding: 20px;
}}

@media (max-width: 1000px) {{
    .sidebar {{
        position: relative;
        width: 100%;
        height: auto;
    }}
    .main {{
        margin-left: 0;
    }}
    .cards, .grid-2 {{
        grid-template-columns: 1fr;
    }}
}}
</style>
</head>

<body>

<div class="sidebar">
    <div class="logo">GitHub Mining</div>
    <div class="subtitle">Dark Modern Graph Analytics Dashboard</div>

    <button class="nav-btn active" onclick="showSection('overview', this)">Overview</button>
    <button class="nav-btn" onclick="showSection('pagerank', this)">PageRank Analysis</button>
    <button class="nav-btn" onclick="showSection('centrality', this)">Degree Centrality</button>
    <button class="nav-btn" onclick="showSection('hits', this)">HITS Analysis</button>
    <button class="nav-btn" onclick="showSection('network', this)">Network Graph</button>
    <button class="nav-btn" onclick="showSection('table', this)">Top Table</button>
</div>

<div class="main">

<div class="header">
    <div>
        <h1>GitHub Repository Mining Dashboard</h1>
        <p>Graph Construction, PageRank, HITS, Centrality, and Repository Influence Analysis</p>
    </div>
    <div class="badge">Interactive HTML + CSS + JS</div>
</div>

<div class="cards">
    <div class="card">
        <p>Total Repositories</p>
        <h2>{len(df)}</h2>
    </div>
    <div class="card">
        <p>Highest PageRank</p>
        <h2>{df['pagerank_score'].max():.6f}</h2>
    </div>
    <div class="card">
        <p>Highest Degree</p>
        <h2>{df['degree_centrality'].max():.4f}</h2>
    </div>
    <div class="card">
        <p>Highest Authority</p>
        <h2>{df['authority_score'].max():.6f}</h2>
    </div>
</div>

<div id="overview" class="section active">
    <div class="grid-2">
        <div class="panel">
            <h2>Project Objective</h2>
            <div class="insight">
                This dashboard analyzes GitHub repositories as a graph network.
                Each repository is a node, and edges represent relationships based on shared technologies or topics.
                The goal is to identify influential and central repositories using graph mining algorithms.
            </div>
        </div>

        <div class="panel">
            <h2>Top 1 Repository Summary</h2>
            <div class="top-card">
                <h3>{top1_pagerank['repository']}</h3>
                <p>
                This repository has the highest PageRank score:
                <b>{top1_pagerank['pagerank_score']:.6f}</b>.
                It is considered the most influential repository because it is connected to other important repositories.
                </p>
            </div>
        </div>
    </div>

    <div class="grid-2">
        <div class="panel">
            {pio.to_html(fig_scatter, full_html=False, include_plotlyjs=False)}
            <div class="insight">
                This chart compares Degree Centrality and PageRank.
                If the points move upward together, it means highly connected repositories are also highly influential.
            </div>
        </div>

        <div class="panel">
            {pio.to_html(fig_corr, full_html=False, include_plotlyjs=False)}
            <div class="insight">
                The correlation matrix shows how strongly the graph metrics are related.
            </div>
        </div>
    </div>
</div>

<div id="pagerank" class="section">
    <div class="panel">
        {pio.to_html(fig_pagerank, full_html=False, include_plotlyjs=False)}
        <div class="insight">
            <b>PageRank</b> measures repository influence based on the importance of its connections.
            The highest PageRank repository is <b>{top1_pagerank['repository']}</b>.
        </div>
    </div>
</div>

<div id="centrality" class="section">
    <div class="panel">
        {pio.to_html(fig_degree, full_html=False, include_plotlyjs=False)}
        <div class="insight">
            <b>Degree Centrality</b> measures how many direct connections a repository has.
            The most connected repository is <b>{top1_degree['repository']}</b>.
        </div>
    </div>
</div>

<div id="hits" class="section">
    <div class="panel">
        {pio.to_html(fig_hits, full_html=False, include_plotlyjs=False)}
        <div class="insight">
            <b>HITS</b> separates repositories into hubs and authorities.
            The highest hub is <b>{top1_hub['repository']}</b>, while the highest authority is <b>{top1_authority['repository']}</b>.
        </div>
    </div>
</div>

<div id="network" class="section">
    <div class="panel">
        {pio.to_html(fig_network, full_html=False, include_plotlyjs=False)}
        <div class="insight">
            This interactive graph shows top repositories as nodes.
            Edges represent relationships between repositories.
            Node color represents PageRank importance.
        </div>
    </div>
</div>

<div id="table" class="section">
    <div class="panel">
        <h2>Top Repositories Table</h2>
        <input class="search-box" type="text" id="searchInput" onkeyup="searchTable()" placeholder="Search repository name...">

        <table id="repoTable">
            <thead>
                <tr>
                    <th>Repository</th>
                    <th>Degree Centrality</th>
                    <th>PageRank</th>
                    <th>Hub Score</th>
                    <th>Authority Score</th>
                </tr>
            </thead>
            <tbody>
                {table_rows}
            </tbody>
        </table>
    </div>
</div>

<div class="footer">
    Data Mining Project | GitHub Repository Graph Construction & Link Analysis
</div>

</div>

<script>
function showSection(sectionId, button) {{
    const sections = document.querySelectorAll('.section');
    sections.forEach(sec => sec.classList.remove('active'));

    document.getElementById(sectionId).classList.add('active');

    const buttons = document.querySelectorAll('.nav-btn');
    buttons.forEach(btn => btn.classList.remove('active'));

    button.classList.add('active');
}}

function searchTable() {{
    let input = document.getElementById("searchInput");
    let filter = input.value.toLowerCase();
    let table = document.getElementById("repoTable");
    let rows = table.getElementsByTagName("tr");

    for (let i = 1; i < rows.length; i++) {{
        let repoCell = rows[i].getElementsByTagName("td")[0];
        if (repoCell) {{
            let txtValue = repoCell.textContent || repoCell.innerText;
            rows[i].style.display = txtValue.toLowerCase().includes(filter) ? "" : "none";
        }}
    }}
}}
</script>

</body>
</html>
"""

with open("dark_modern_github_dashboard.html", "w", encoding="utf-8") as f:
    f.write(html)

print("Dark modern dashboard created: dark_modern_github_dashboard.html")

Dark modern dashboard created: dark_modern_github_dashboard.html


In [4]:
from google.colab import files
files.download("dark_modern_github_dashboard.html")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>